In [1]:
# csv 다운로드
from google.colab import files

uploaded = files.upload()

Saving H1_방문자수_폐업률_결합_2023_2025.csv to H1_방문자수_폐업률_결합_2023_2025.csv


In [18]:
# 한글 폰트 설치
!apt-get -qq install fonts-nanum

Selecting previously unselected package fonts-nanum.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


In [19]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# 나눔고딕 폰트 경로
font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# matplotlib 기본 폰트 설정
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rc("font", family=font_name)

# 마이너스(-) 기호 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

print("설정된 폰트:", font_name)

설정된 폰트: NanumGothic


## 라이브러리
pandas	CSV 읽기, 데이터 전처리  
numpy	수치 계산  
matplotlib	그래프  
seaborn	통계 그래프  
scipy.stats	Pearson, Spearman 상관분석  
statsmodels	회귀분석

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from scipy.stats import spearmanr

import statsmodels.api as sm

## csv 파일 확인

In [3]:
# csv 파일 읽기
file_name = "H1_방문자수_폐업률_결합_2023_2025.csv"

df = pd.read_csv(file_name, encoding="utf-8-sig")

# 잘읽었는지 확인
print("데이터 불러오기 완료")
print("행 수:", len(df))
print("열 수:", len(df.columns))

데이터 불러오기 완료
행 수: 48
열 수: 8


In [4]:
df

,연도,행정구,현지인 방문자 수,외지인 방문자 수,외국인 방문자 수,폐업수,사업체수,폐업률
0,2023,강서구,30254373,39971333,2704399,3755,9501,39.52
1,2023,금정구,44717001,29679346,321529,3645,12451,29.27
2,2023,기장군,35402889,41591226,1156329,3429,10411,32.94
3,2023,남구,50914948,30066424,1169844,4104,12825,32.00
4,2023,동구,16910425,33045121,1347927,2212,6680,33.11
5,2023,동래구,49329933,36786089,171791,4195,15140,27.71
6,2023,부산진구,76970831,71357781,1498406,7057,23860,29.58
7,2023,북구,55222330,23854599,160439,3486,11448,30.45
8,2023,사상구,33470622,27717210,345744,3927,10986,35.75
9,2023,사하구,51782610,23240233,980744,4102,13744,29.85


In [5]:
# 컬럼 이름 확인
print(df.columns.tolist())

['연도', '행정구', '현지인 방문자 수', '외지인 방문자 수', '외국인 방문자 수', '폐업수', '사업체수', '폐업률']


In [6]:
# 데이터 자료형 확인
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   연도         48 non-null     int64  
 1   행정구        48 non-null     object 
 2   현지인 방문자 수  48 non-null     int64  
 3   외지인 방문자 수  48 non-null     int64  
 4   외국인 방문자 수  48 non-null     int64  
 5   폐업수        48 non-null     int64  
 6   사업체수       48 non-null     int64  
 7   폐업률        48 non-null     float64
dtypes: float64(1), int64(6), object(1)
memory usage: 3.1+ KB


In [7]:
# 결측치 확인
print(df.isnull().sum())

연도           0
행정구          0
현지인 방문자 수    0
외지인 방문자 수    0
외국인 방문자 수    0
폐업수          0
사업체수         0
폐업률          0
dtype: int64


In [8]:
# 중복하는 값이 있는지 확인
print("전체 중복 행:", df.duplicated().sum())
print("연도 + 행정구 중복:", df.duplicated(["연도", "행정구"]).sum())

전체 중복 행: 0
연도 + 행정구 중복: 0


In [9]:
# 연도별 데이터 개수 확인
print(df.groupby("연도").size())

연도
2023    16
2024    16
2025    16
dtype: int64


In [10]:
# 행정구별 데이터 개수 확인
print(df.groupby("행정구").size())

행정구
강서구     3
금정구     3
기장군     3
남구      3
동구      3
동래구     3
부산진구    3
북구      3
사상구     3
사하구     3
서구      3
수영구     3
연제구     3
영도구     3
중구      3
해운대구    3
dtype: int64


In [11]:
# 분석에 사용할 변수만 확인
analysis_cols = [
    "현지인 방문자 수",
    "외지인 방문자 수",
    "외국인 방문자 수",
    "폐업률"
]

df[analysis_cols].head()

,현지인 방문자 수,외지인 방문자 수,외국인 방문자 수,폐업률
0,30254373,39971333,2704399,39.52
1,44717001,29679346,321529,29.27
2,35402889,41591226,1156329,32.94
3,50914948,30066424,1169844,32.00
4,16910425,33045121,1347927,33.11


In [12]:
# 기초통계량 확인
df[analysis_cols].describe()

,현지인 방문자 수,외지인 방문자 수,외국인 방문자 수,폐업률
count,4.800000e+01,4.800000e+01,4.800000e+01,48.000000
mean,4.058811e+07,3.690081e+07,1.479781e+06,30.693542
std,2.042873e+07,1.506901e+07,1.169776e+06,3.311045
min,1.415139e+07,1.844996e+07,1.604390e+05,26.650000
25%,2.264026e+07,2.715940e+07,4.285765e+05,28.645000
50%,3.523591e+07,3.285739e+07,1.344942e+06,29.600000
75%,5.113186e+07,4.085473e+07,2.005899e+06,32.090000
max,8.932198e+07,7.621595e+07,5.571300e+06,42.280000


In [13]:
# 방문자 수의 단위 변경
# 회귀분석 해석에 용이하게 하기 위해(백만명 단위로)
df["현지인 방문자 수_백만"] = df["현지인 방문자 수"] / 1_000_000
df["외지인 방문자 수_백만"] = df["외지인 방문자 수"] / 1_000_000
df["외국인 방문자 수_백만"] = df["외국인 방문자 수"] / 1_000_000

In [14]:
# 분석용 변수 확인
analysis_cols_scaled = [
    "현지인 방문자 수_백만",
    "외지인 방문자 수_백만",
    "외국인 방문자 수_백만",
    "폐업률"
]

df[analysis_cols_scaled].describe()

,현지인 방문자 수_백만,외지인 방문자 수_백만,외국인 방문자 수_백만,폐업률
count,48.000000,48.000000,48.000000,48.000000
mean,40.588109,36.900813,1.479781,30.693542
std,20.428735,15.069009,1.169776,3.311045
min,14.151391,18.449959,0.160439,26.650000
25%,22.640264,27.159399,0.428576,28.645000
50%,35.235906,32.857389,1.344942,29.600000
75%,51.131864,40.854730,2.005899,32.090000
max,89.321976,76.215951,5.571300,42.280000


#Pearson 상관 분석
> 방문자 수와 폐업률 사이의 관계가 있는가?  
-> 두 변수 사이의 선형적인 관계의 방향과 강도 측정

###1. Pearson 상관계수
> Pearson 상관계수가 -: 음의 상관계수(방문자 수 증가 -> 폐업률 감소)  
Pearson 상관계수가 +: 양의 상관계수(방문자 수 증가 -> 폐업률 증가)  
Pearson 상관계수가 0에 가까움: 유의미한 결과 x

###2. p-value 값(일반적 기준 0.05)
> p-value < 0.05 -> 통계적으로 유의미한 관계  
p-value >= 0.05  -> 통계적으로 유의미하지 않은 관계

In [16]:
# Pearson 상관분석
variables = [
    "현지인 방문자 수_백만",
    "외지인 방문자 수_백만",
    "외국인 방문자 수_백만"
]

results = []

for variable in variables:
    r, p = pearsonr(df[variable], df["폐업률"])

    results.append({
        "방문자 유형": variable,
        "Pearson 상관계수": r,
        "p-value": p
    })

pearson_result = pd.DataFrame(results)

pearson_result

,방문자 유형,Pearson 상관계수,p-value
0,현지인 방문자 수_백만,-0.028659,0.846674
1,외지인 방문자 수_백만,0.166938,0.256765
2,외국인 방문자 수_백만,0.501917,0.000278


In [17]:
# 변수의 상관관계 확인
correlation_matrix = df[
    [
        "현지인 방문자 수_백만",
        "외지인 방문자 수_백만",
        "외국인 방문자 수_백만",
        "폐업률"
    ]
].corr()

correlation_matrix

,현지인 방문자 수_백만,외지인 방문자 수_백만,외국인 방문자 수_백만,폐업률
현지인 방문자 수_백만,1.000000,0.647501,0.090198,-0.028659
외지인 방문자 수_백만,0.647501,1.000000,0.475413,0.166938
외국인 방문자 수_백만,0.090198,0.475413,1.000000,0.501917
폐업률,-0.028659,0.166938,0.501917,1.000000


# Spearman 분석
> Pearson 분석의 결과가 다른 요인때문에 영향을 받았을 가능성을 추가로 확인할 필요
Spearman은 각 데이터의 순위를 매겨 일부 극단적인 값이나 데이터의 비선형적인 구조가 Pearson 결과에 영향을 주었을 가능성을 검토할 수 있다.

>Pearson 과 Spearman의 결과가 비슷하다면: 특정한 값에 의해 나타난 결과일 가능성이 낮다  
Pearson과 Spearman의 결과가 크게 다르면: 일부 극단적인 값이나 데이터의 비선형적인 구조가 Pearson 결과에 영향을 주었을 가능성을 검토해봐야한다.

In [22]:
# spearman 분석
variables = [
    "현지인 방문자 수_백만",
    "외지인 방문자 수_백만",
    "외국인 방문자 수_백만"
]

spearman_results = []

for variable in variables:
    rho, p = spearmanr(
        df[variable],
        df["폐업률"]
    )

    spearman_results.append({
        "방문자 유형": variable,
        "Spearman 상관계수": rho,
        "p-value": p
    })

spearman_result = pd.DataFrame(spearman_results)

spearman_result

,방문자 유형,Spearman 상관계수,p-value
0,현지인 방문자 수_백만,-0.020138,0.891936
1,외지인 방문자 수_백만,0.129729,0.379499
2,외국인 방문자 수_백만,0.299083,0.038922


In [23]:
# 현지인 방문자 수와 폐업률 사이에는 뚜렷한 선형적 또는 순위 기반 관계가 확인되지 않았다.
# 외지인 방문자 수가 증가하는 방향과 폐업률이 증가하는 방향이 어느 정도 같이 움직이는 경향은 관찰되지만, 통계적으로 유의한 관계라고 볼 수는 없다.
# 외국인 방문자 수와 폐업률의 관계가 존재한다는 결과 자체는 두 분석에서 일치하지만, 관계의 강도는 동일하지 않다.

# 결론
# 외국인 방문자 수와 폐업률 사이에는 통계적으로 유의한 양의 관계가 확인되었다.
# Pearson 상관분석에서는 r=0.502(p<0.001), Spearman 순위상관분석에서는 ρ=0.299(p=0.039)로 나타났다.

# VIF - Variance Inflation Factor(분산팽창계수)
>다중공선성이 있는지를 확인하는 분석  
단순히 상관계수만 보고 다중공선성이 있다고 확정할 수 없음  
->VIF: 어떤 독립변수의 정보가 다른 독립변수들과 얼마나 겹치는지를 숫자로 나타낸 것

>VIF가 높을수록 해당 변수의 회귀계수를 안정적으로 추정하기 어려워진다.  
1에 가까움:	다른 독립변수와 거의 겹치지 않음  
1-5:	일반적으로 문제 없음  
5-10:	다중공선성 주의  
10 이상:	심각한 다중공선성 의심  

In [24]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 회귀분석에 사용할 독립변수만 선택
X_vif = df[
    [
        "현지인 방문자 수_백만",
        "외지인 방문자 수_백만",
        "외국인 방문자 수_백만"
    ]
]

# VIF 결과를 저장할 데이터프레임 생성
vif_result = pd.DataFrame()

# 변수명 저장
vif_result["변수"] = X_vif.columns

# 각 독립변수의 VIF 계산
vif_result["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

# 결과 출력
vif_result

,변수,VIF
0,현지인 방문자 수_백만,9.215901
1,외지인 방문자 수_백만,14.632177
2,외국인 방문자 수_백만,3.757266


In [25]:
# 현지인·외지인 변수의 다중공선성 문제가 확인됨 = 현지인 방문자 수와 외지인 방문자 수가 서로 비슷한 정보를 가지고 있다.
# 현지인과 외지인이 서로 비슷하게 움직이면 모델 입장에서는:
# "이 폐업률 변화가 현지인 방문자 때문인지, 외지인 방문자 때문인지"구분하기 어려워진다.

# 그 결과 회귀분석에서:

# 회귀계수가 불안정해질 수 있음
# 표준오차가 커질 수 있음
# p-value가 커질 수 있음
# 계수의 방향이 예상과 다르게 나타날 수 있음
# 변수별 효과를 해석하기 어려워질 수 있음

# 등의 문제가 생긴다.

# 단순선형회귀분석
> 각 독립변수를 독립적으로 확인 -> 단순선형회귀를 통해 관계의 방향과 크기를 확인

>coef가 +: 방문자수 증가와 폐업률 증가 방향  
coef가 -: 방문자수 증감와 폐업률 감소 방향  
P>|t|: p-value와 동일  
R-squared: 결정계수 R**2 -> 이 회귀모형이 폐업률의 변동을 어느 정도 설명하는가?

In [28]:
# 현지인 방문자 수 -> 폐업률 분석

# 독립변수(X)
X = df[["현지인 방문자 수_백만"]]

# 종속변수(Y)
y = df["폐업률"]

# 상수항 추가
X = sm.add_constant(X)

# 단순선형회귀모형 생성 및 적합
model_local = sm.OLS(y, X).fit()

# 회귀분석 결과 출력
print(model_local.summary())

                            OLS Regression Results                            
Dep. Variable:                    폐업률   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.021
Method:                 Least Squares   F-statistic:                   0.03781
Date:                Tue, 18 Aug 2026   Prob (F-statistic):              0.847
Time:                        06:17:09   Log-Likelihood:                -125.05
No. Observations:                  48   AIC:                             254.1
Df Residuals:                      46   BIC:                             257.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           30.8821      1.083     28.512   

In [30]:
# 외지인 방문자 수 -> 폐업률

# 독립변수(X)
X = df[["외지인 방문자 수_백만"]]

# 종속변수(Y)
y = df["폐업률"]

# 상수항 추가
X = sm.add_constant(X)

# 단순선형회귀모형
model_outside = sm.OLS(y, X).fit()

# 결과 출력
print(model_outside.summary())

                            OLS Regression Results                            
Dep. Variable:                    폐업률   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     1.319
Date:                Tue, 18 Aug 2026   Prob (F-statistic):              0.257
Time:                        06:17:38   Log-Likelihood:                -124.39
No. Observations:                  48   AIC:                             252.8
Df Residuals:                      46   BIC:                             256.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           29.3400      1.271     23.079   

In [31]:
# 외국인 방문자 수 -> 폐업률

# 독립변수(X)
X = df[["외국인 방문자 수_백만"]]

# 종속변수(Y)
y = df["폐업률"]

# 상수항 추가
X = sm.add_constant(X)

# 단순선형회귀모형
model_foreign = sm.OLS(y, X).fit()

# 결과 출력
print(model_foreign.summary())

                            OLS Regression Results                            
Dep. Variable:                    폐업률   R-squared:                       0.252
Model:                            OLS   Adj. R-squared:                  0.236
Method:                 Least Squares   F-statistic:                     15.49
Date:                Tue, 18 Aug 2026   Prob (F-statistic):           0.000278
Time:                        06:18:00   Log-Likelihood:                -118.11
No. Observations:                  48   AIC:                             240.2
Df Residuals:                      46   BIC:                             244.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           28.5913      0.678     42.161   

In [32]:
# 세 모델의 결과를 하나의 표로 정리

regression_results = []

models = {
    "현지인 방문자 수": model_local,
    "외지인 방문자 수": model_outside,
    "외국인 방문자 수": model_foreign
}

for name, model in models.items():
    regression_results.append({
        "방문자 유형": name,
        "회귀계수": model.params.iloc[1],
        "p-value": model.pvalues.iloc[1],
        "R²": model.rsquared,
        "Adj. R²": model.rsquared_adj
    })

regression_result = pd.DataFrame(regression_results)

regression_result

,방문자 유형,회귀계수,p-value,R²,Adj. R²
0,현지인 방문자 수,-0.004645,0.846674,0.000821,-0.020900
1,외지인 방문자 수,0.036680,0.256765,0.027868,0.006735
2,외국인 방문자 수,1.420673,0.000278,0.251920,0.235658


# 다중선형회귀분석

In [33]:
# 독립변수
X = df[
    [
        "현지인 방문자 수_백만",
        "외지인 방문자 수_백만",
        "외국인 방문자 수_백만"
    ]
]

# 종속변수
y = df["폐업률"]

In [34]:
# 상수항 추가
X = sm.add_constant(X)

In [35]:
model_multiple = sm.OLS(y, X).fit()

In [36]:
print(model_multiple.summary())

                            OLS Regression Results                            
Dep. Variable:                    폐업률   R-squared:                       0.259
Model:                            OLS   Adj. R-squared:                  0.209
Method:                 Least Squares   F-statistic:                     5.131
Date:                Tue, 18 Aug 2026   Prob (F-statistic):            0.00395
Time:                        06:24:51   Log-Likelihood:                -117.87
No. Observations:                  48   AIC:                             243.7
Df Residuals:                      44   BIC:                             251.2
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           29.2068      1.164     25.085   

In [37]:
multiple_result = pd.DataFrame({
    "회귀계수": model_multiple.params,
    "표준오차": model_multiple.bse,
    "t값": model_multiple.tvalues,
    "p-value": model_multiple.pvalues,
    "하한 95%": model_multiple.conf_int()[0],
    "상한 95%": model_multiple.conf_int()[1]
})

multiple_result

,회귀계수,표준오차,t값,p-value,하한 95%,상한 95%
const,29.206847,1.164296,25.085419,1.065229e-27,26.860363,31.553331
현지인 방문자 수_백만,-0.005595,0.029177,-0.191769,8.488055e-01,-0.064398,0.053207
외지인 방문자 수_백만,-0.014424,0.044777,-0.322120,7.488873e-01,-0.104667,0.075819
외국인 방문자 수_백만,1.517821,0.441374,3.438852,1.289156e-03,0.628290,2.407353


In [38]:
print("R² :", model_multiple.rsquared)
print("수정 R² :", model_multiple.rsquared_adj)

R² : 0.2591779477030729
수정 R² : 0.20866735322828245


In [39]:
print("F-statistic :", model_multiple.fvalue)
print("F-test p-value :", model_multiple.f_pvalue)

F-statistic : 5.131160113992066
F-test p-value : 0.003947629088730004


방문자 수 전체가 폐업률과 일괄적으로 유의한 관계를 가지는 것은 아니다. 방문자 유형별로 차이가 나타났다. 현지인과 외지인 방문자 수에서는 유의한 관계가 확인되지 않았지만, 외국인 방문자 수에서는 폐업률과 일관된 양의 관계가 확인되었다. 이 관계는 다른 방문자 유형을 통제한 다중선형회귀에서도 통계적으로 유의했다.

In [40]:
# 잔차 정규성 + 등분산성 확인

from scipy.stats import shapiro
from statsmodels.stats.diagnostic import het_breuschpagan

# 잔차
residuals = model_multiple.resid

# ① Shapiro-Wilk 정규성 검정
shapiro_stat, shapiro_p = shapiro(residuals)

# ② Breusch-Pagan 등분산성 검정
bp_result = het_breuschpagan(residuals, model_multiple.model.exog)

bp_lm_stat, bp_lm_p, bp_f_stat, bp_f_p = bp_result

print("===== 회귀진단 결과 =====")

print("\n[Shapiro-Wilk 정규성 검정]")
print("통계량 :", shapiro_stat)
print("p-value :", shapiro_p)

print("\n[Breusch-Pagan 등분산성 검정]")
print("LM 통계량 :", bp_lm_stat)
print("LM p-value :", bp_lm_p)
print("F 통계량 :", bp_f_stat)
print("F p-value :", bp_f_p)

===== 회귀진단 결과 =====

[Shapiro-Wilk 정규성 검정]
통계량 : 0.9341360807051835
p-value : 0.009716748898585294

[Breusch-Pagan 등분산성 검정]
LM 통계량 : 6.418304423443924
LM p-value : 0.09294065839654994
F 통계량 : 2.263859860416996
F p-value : 0.09426669350563512


In [41]:
# 이상치/영향력 관측치 확인

# 회귀모형의 영향력 진단
influence = model_multiple.get_influence()

# Cook's Distance
cooks_d = influence.cooks_distance[0]

# 가장 영향력이 큰 관측치 5개
top5 = pd.DataFrame({
    "행 번호": range(len(cooks_d)),
    "Cook's Distance": cooks_d
}).sort_values(
    "Cook's Distance",
    ascending=False
).head(5)

print("===== 영향력 관측치 상위 5개 =====")
display(top5)

===== 영향력 관측치 상위 5개 =====


,행 번호,Cook's Distance
16,16,0.319714
15,15,0.153903
47,47,0.140175
46,46,0.094838
32,32,0.090246


In [42]:
# Cook's Distance가 높은 관측치 확인

influential_indices = [16, 15, 47, 46, 32]

df.loc[
    influential_indices,
    [
        "연도",
        "행정구",
        "현지인 방문자 수_백만",
        "외지인 방문자 수_백만",
        "외국인 방문자 수_백만",
        "폐업률"
    ]
]

,연도,행정구,현지인 방문자 수_백만,외지인 방문자 수_백만,외국인 방문자 수_백만,폐업률
16,2024,강서구,30.329530,41.067382,4.030043,42.28
15,2023,해운대구,86.758205,67.603182,2.004682,36.04
47,2025,해운대구,89.321976,72.850929,4.313576,30.86
46,2025,중구,14.353420,40.783846,2.756611,27.71
32,2025,강서구,33.443717,44.556991,5.571300,38.96


#행정구 + 연도 고정

In [43]:
# 행정구와 연도를 범주형 변수로 변환
df["행정구"] = df["행정구"].astype(str)
df["연도"] = df["연도"].astype(str)

print("행정구 수:", df["행정구"].nunique())
print("연도 수:", df["연도"].nunique())

print("\n행정구:")
print(df["행정구"].unique())

print("\n연도:")
print(df["연도"].unique())

행정구 수: 16
연도 수: 3

행정구:
['강서구' '금정구' '기장군' '남구' '동구' '동래구' '부산진구' '북구' '사상구' '사하구' '서구' '수영구'
 '연제구' '영도구' '중구' '해운대구']

연도:
['2023' '2024' '2025']


In [44]:
# 독립변수
X_fe = df[
    [
        "현지인 방문자 수_백만",
        "외지인 방문자 수_백만",
        "외국인 방문자 수_백만"
    ]
].copy()

# 행정구 고정효과
district_dummies = pd.get_dummies(
    df["행정구"],
    prefix="행정구",
    drop_first=True
)

# 연도 고정효과
year_dummies = pd.get_dummies(
    df["연도"],
    prefix="연도",
    drop_first=True
)

# 독립변수 + 행정구 효과 + 연도 효과 결합
X_fe = pd.concat(
    [
        X_fe,
        district_dummies,
        year_dummies
    ],
    axis=1
)

# 상수항 추가
X_fe = sm.add_constant(X_fe)

# 종속변수
y_fe = df["폐업률"]

# 숫자형으로 변환
X_fe = X_fe.astype(float)
y_fe = y_fe.astype(float)

# 고정효과 회귀 실행
model_fe = sm.OLS(y_fe, X_fe).fit()

# 결과 출력
print(model_fe.summary())

                            OLS Regression Results                            
Dep. Variable:                    폐업률   R-squared:                       0.933
Model:                            OLS   Adj. R-squared:                  0.883
Method:                 Least Squares   F-statistic:                     18.75
Date:                Tue, 18 Aug 2026   Prob (F-statistic):           5.24e-11
Time:                        06:39:59   Log-Likelihood:                -60.254
No. Observations:                  48   AIC:                             162.5
Df Residuals:                      27   BIC:                             201.8
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           48.7672      8.432      5.783   

In [45]:
# 핵심 결과 보기 쉽게 정리

# 방문자 변수만 추출
fe_result = pd.DataFrame({
    "회귀계수": model_fe.params,
    "표준오차": model_fe.bse,
    "t값": model_fe.tvalues,
    "p-value": model_fe.pvalues,
    "하한 95%": model_fe.conf_int()[0],
    "상한 95%": model_fe.conf_int()[1]
})

# 방문자 변수만 출력
fe_result = fe_result.loc[
    [
        "현지인 방문자 수_백만",
        "외지인 방문자 수_백만",
        "외국인 방문자 수_백만"
    ]
]

display(fe_result)

,회귀계수,표준오차,t값,p-value,하한 95%,상한 95%
현지인 방문자 수_백만,-0.019313,0.191795,-0.100698,0.920534,-0.412845,0.374218
외지인 방문자 수_백만,-0.165479,0.215348,-0.768425,0.448904,-0.607337,0.276379
외국인 방문자 수_백만,-0.121745,0.564145,-0.215805,0.830762,-1.279275,1.035785


In [46]:
print("R² :", model_fe.rsquared)
print("수정 R² :", model_fe.rsquared_adj)

print("F-statistic :", model_fe.fvalue)
print("F-test p-value :", model_fe.f_pvalue)

R² : 0.9328449896836319
수정 R² : 0.8831005375974333
F-statistic : 18.752744287285985
F-test p-value : 5.2429975664756286e-11


일반적인 상관분석과 OLS에서는 외국인 방문자 수와 폐업률 사이에 유의한 양의 관계가 나타났지만, 행정구 및 연도 고정효과를 통제한 분석에서는 그 관계가 통계적으로 유의하지 않았다.

단순 상관관계 수준에서는 외국인 방문자 수와 폐업률 사이에 유의한 양의 관계가 나타났다. 그러나 행정구와 연도에 따른 고유한 차이를 통제한 고정효과 회귀에서는 현지인·외지인·외국인 방문자 모두 폐업률과 유의한 관계를 나타내지 않았다. 따라서 현재 데이터만으로는 방문자 수가 폐업률에 독립적으로 유의한 영향을 미친다고 판단하기 어렵다.

# 시각화 그래프 만들기

In [87]:
# ============================================================
# 1단계. Plotly 시각화 기본 설정 및 데이터 확인
# ============================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# Plotly 기본 렌더링 설정
import plotly.io as pio

pio.renderers.default = "colab"


# ------------------------------------------------------------
# 분석에 필요한 컬럼
# ------------------------------------------------------------

required_cols = [
    "연도",
    "행정구",
    "현지인 방문자 수_백만",
    "외지인 방문자 수_백만",
    "외국인 방문자 수_백만",
    "폐업률"
]


# ------------------------------------------------------------
# 컬럼 존재 여부 확인
# ------------------------------------------------------------

missing_cols = [
    col
    for col in required_cols
    if col not in df.columns
]


if len(missing_cols) > 0:

    print("❌ 다음 컬럼이 없습니다.")

    for col in missing_cols:
        print("-", col)

    raise ValueError(
        "컬럼명을 확인해주세요."
    )


print("✅ 필요한 컬럼이 모두 존재합니다.")


# ------------------------------------------------------------
# 분석용 데이터 복사
# ------------------------------------------------------------

analysis_df = df[
    required_cols
].copy()


# ------------------------------------------------------------
# 데이터 타입 정리
# ------------------------------------------------------------

analysis_df["연도"] = (
    analysis_df["연도"]
    .astype(str)
)

analysis_df["행정구"] = (
    analysis_df["행정구"]
    .astype(str)
)


numeric_cols = [
    "현지인 방문자 수_백만",
    "외지인 방문자 수_백만",
    "외국인 방문자 수_백만",
    "폐업률"
]


for col in numeric_cols:

    analysis_df[col] = pd.to_numeric(
        analysis_df[col],
        errors="coerce"
    )


# ------------------------------------------------------------
# 결측치 확인
# ------------------------------------------------------------

print()
print("===== 결측치 확인 =====")

print(
    analysis_df[
        required_cols
    ].isnull().sum()
)


# ------------------------------------------------------------
# 결측치 제거
# ------------------------------------------------------------

analysis_df = analysis_df.dropna(
    subset=required_cols
).reset_index(drop=True)


# ------------------------------------------------------------
# 연도 / 행정구 확인
# ------------------------------------------------------------

years = sorted(
    analysis_df["연도"].unique()
)

districts = sorted(
    analysis_df["행정구"].unique()
)


print()
print("===== 데이터 기본 정보 =====")

print(
    "전체 행 수 :",
    len(analysis_df)
)

print(
    "연도 수 :",
    len(years)
)

print(
    "연도 :",
    years
)

print(
    "행정구 수 :",
    len(districts)
)

print(
    "행정구 :",
    districts
)


# ------------------------------------------------------------
# 데이터 미리보기
# ------------------------------------------------------------

print()
print("===== 데이터 미리보기 =====")

display(
    analysis_df.head()
)

✅ 필요한 컬럼이 모두 존재합니다.

===== 결측치 확인 =====
연도              0
행정구             0
현지인 방문자 수_백만    0
외지인 방문자 수_백만    0
외국인 방문자 수_백만    0
폐업률             0
dtype: int64

===== 데이터 기본 정보 =====
전체 행 수 : 48
연도 수 : 3
연도 : ['2023', '2024', '2025']
행정구 수 : 16
행정구 : ['강서구', '금정구', '기장군', '남구', '동구', '동래구', '부산진구', '북구', '사상구', '사하구', '서구', '수영구', '연제구', '영도구', '중구', '해운대구']

===== 데이터 미리보기 =====


,연도,행정구,현지인 방문자 수_백만,외지인 방문자 수_백만,외국인 방문자 수_백만,폐업률
0,2023,강서구,30.254373,39.971333,2.704399,39.52
1,2023,금정구,44.717001,29.679346,0.321529,29.27
2,2023,기장군,35.402889,41.591226,1.156329,32.94
3,2023,남구,50.914948,30.066424,1.169844,32.00
4,2023,동구,16.910425,33.045121,1.347927,33.11


In [88]:
# ============================================================
# 2단계. Pearson 산점도 3개
# ============================================================

import plotly.graph_objects as go


# ============================================================
# 1. 행정구별 색상
# ============================================================

color_list = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
    "#bcbd22",
    "#17becf",
    "#393b79",
    "#637939",
    "#8c6d31",
    "#843c39",
    "#7b4173",
    "#3182bd"
]


district_colors = {}

for i, district in enumerate(districts):

    district_colors[district] = (
        color_list[
            i % len(color_list)
        ]
    )


# ============================================================
# 2. 연도별 마커 모양
# ============================================================

year_symbols = {

    "2023": "circle",

    "2024": "square",

    "2025": "diamond"

}


# ============================================================
# 3. 방문자 변수
# ============================================================

visitor_vars = [

    "현지인 방문자 수_백만",

    "외지인 방문자 수_백만",

    "외국인 방문자 수_백만"

]


# ============================================================
# 4. Pearson 그래프 생성 함수
# ============================================================

def make_pearson_graph(variable):

    fig = go.Figure()


    # --------------------------------------------------------
    # 행정구 × 연도별 데이터
    # --------------------------------------------------------

    for year in years:

        for district in districts:

            temp = analysis_df[
                (
                    analysis_df["연도"]
                    == year
                )
                &
                (
                    analysis_df["행정구"]
                    == district
                )
            ]


            # 해당 데이터가 없으면 넘어감
            if len(temp) == 0:

                continue


            # ------------------------------------------------
            # 마우스 오버 정보
            # ------------------------------------------------

            hover_text = []


            for _, row in temp.iterrows():

                text = (

                    "연도: "
                    + str(row["연도"])

                    + "<br>"

                    + "행정구: "
                    + str(row["행정구"])

                    + "<br>"

                    + variable
                    + ": "
                    + format(
                        row[variable],
                        ".3f"
                    )
                    + " 백만"

                    + "<br>"

                    + "폐업률: "
                    + format(
                        row["폐업률"],
                        ".2f"
                    )
                    + "%"

                )

                hover_text.append(text)


            # ------------------------------------------------
            # 그래프 점
            # ------------------------------------------------

            fig.add_trace(

                go.Scatter(

                    x=temp[
                        variable
                    ],

                    y=temp[
                        "폐업률"
                    ],

                    mode="markers",

                    name=district,

                    legendgroup=district,

                    # 첫 번째 연도만 범례에 표시
                    showlegend=(
                        year
                        ==
                        years[0]
                    ),

                    marker=dict(

                        size=18,

                        color=district_colors[
                            district
                        ],

                        symbol=year_symbols[
                            year
                        ],

                        line=dict(

                            color="black",

                            width=1

                        )

                    ),

                    text=hover_text,

                    hovertemplate=
                        "%{text}"
                        +
                        "<extra></extra>"

                )

            )


    # ========================================================
    # 5. 전체 데이터 기준 축 범위
    # ========================================================

    xmin = analysis_df[
        variable
    ].min()

    xmax = analysis_df[
        variable
    ].max()


    x_margin = (
        xmax - xmin
    ) * 0.08


    if x_margin == 0:

        x_margin = 1


    ymin = analysis_df[
        "폐업률"
    ].min()

    ymax = analysis_df[
        "폐업률"
    ].max()


    y_margin = (
        ymax - ymin
    ) * 0.08


    if y_margin == 0:

        y_margin = 1


    # ========================================================
    # 6. 그래프 설정
    # ========================================================

    fig.update_layout(

        title=(
            "Pearson 상관분석<br>"
            "<sup>"
            + variable
            + " ↔ 폐업률"
            + "</sup>"
        ),

        xaxis=dict(

            title=variable,

            range=[
                xmin - x_margin,
                xmax + x_margin
            ],

            autorange=False

        ),

        yaxis=dict(

            title="폐업률 (%)",

            range=[
                ymin - y_margin,
                ymax + y_margin
            ],

            autorange=False

        ),

        height=650,

        template="plotly_white",

        font=dict(

            family=(
                "Malgun Gothic, "
                "Noto Sans KR, "
                "Arial"
            )

        ),

        legend=dict(

            title="행정구",

            orientation="v"

        ),

        hoverlabel=dict(

            font_size=14

        )

    )


    return fig


# ============================================================
# 7. Pearson 그래프 3개 생성
# ============================================================

pearson_figures = {}


for variable in visitor_vars:

    pearson_figures[
        variable
    ] = make_pearson_graph(
        variable
    )


# ============================================================
# 8. 그래프 출력
# ============================================================

for variable in visitor_vars:

    print()
    print("=" * 60)
    print(
        "Pearson:",
        variable
    )
    print("=" * 60)

    pearson_figures[
        variable
    ].show()


Pearson: 현지인 방문자 수_백만



Pearson: 외지인 방문자 수_백만



Pearson: 외국인 방문자 수_백만


In [89]:
# ============================================================
# 3단계. Spearman 산점도 3개
# ============================================================

import plotly.graph_objects as go

from scipy.stats import spearmanr


# ============================================================
# 1. Spearman 그래프 생성 함수
# ============================================================

def make_spearman_graph(variable):

    fig = go.Figure()


    # --------------------------------------------------------
    # Spearman 상관계수 계산
    # --------------------------------------------------------

    rho, p_value = spearmanr(

        analysis_df[
            variable
        ],

        analysis_df[
            "폐업률"
        ]

    )


    # --------------------------------------------------------
    # 행정구 × 연도별 데이터
    # --------------------------------------------------------

    for year in years:

        for district in districts:

            temp = analysis_df[
                (
                    analysis_df["연도"]
                    == year
                )
                &
                (
                    analysis_df["행정구"]
                    == district
                )
            ]


            if len(temp) == 0:

                continue


            # ------------------------------------------------
            # 마우스 오버 정보
            # ------------------------------------------------

            hover_text = []


            for _, row in temp.iterrows():

                text = (

                    "연도: "
                    + str(row["연도"])

                    + "<br>"

                    + "행정구: "
                    + str(row["행정구"])

                    + "<br>"

                    + variable
                    + ": "
                    + format(
                        row[variable],
                        ".3f"
                    )
                    + " 백만"

                    + "<br>"

                    + "폐업률: "
                    + format(
                        row["폐업률"],
                        ".2f"
                    )
                    + "%"

                )

                hover_text.append(text)


            # ------------------------------------------------
            # 그래프 점
            # ------------------------------------------------

            fig.add_trace(

                go.Scatter(

                    x=temp[
                        variable
                    ],

                    y=temp[
                        "폐업률"
                    ],

                    mode="markers",

                    name=district,

                    legendgroup=district,

                    showlegend=(
                        year
                        ==
                        years[0]
                    ),

                    marker=dict(

                        size=18,

                        color=district_colors[
                            district
                        ],

                        symbol=year_symbols[
                            year
                        ],

                        line=dict(

                            color="black",

                            width=1

                        )

                    ),

                    text=hover_text,

                    hovertemplate=
                        "%{text}"
                        +
                        "<extra></extra>"

                )

            )


    # ========================================================
    # 2. 축 범위
    # ========================================================

    xmin = analysis_df[
        variable
    ].min()

    xmax = analysis_df[
        variable
    ].max()


    x_margin = (
        xmax - xmin
    ) * 0.08


    if x_margin == 0:

        x_margin = 1


    ymin = analysis_df[
        "폐업률"
    ].min()

    ymax = analysis_df[
        "폐업률"
    ].max()


    y_margin = (
        ymax - ymin
    ) * 0.08


    if y_margin == 0:

        y_margin = 1


    # ========================================================
    # 3. 그래프 설정
    # ========================================================

    fig.update_layout(

        title=(
            "Spearman 상관분석<br>"
            "<sup>"
            + variable
            + " ↔ 폐업률"
            + "<br>"
            + "Spearman ρ = "
            + format(
                rho,
                ".4f"
            )
            + " | p-value = "
            + format(
                p_value,
                ".6f"
            )
            + "</sup>"
        ),

        xaxis=dict(

            title=variable,

            range=[
                xmin - x_margin,
                xmax + x_margin
            ],

            autorange=False

        ),

        yaxis=dict(

            title="폐업률 (%)",

            range=[
                ymin - y_margin,
                ymax + y_margin
            ],

            autorange=False

        ),

        height=650,

        template="plotly_white",

        font=dict(

            family=(
                "Malgun Gothic, "
                "Noto Sans KR, "
                "Arial"
            )

        ),

        legend=dict(

            title="행정구"

        ),

        hoverlabel=dict(

            font_size=14

        )

    )


    return fig


# ============================================================
# 4. Spearman 그래프 3개 생성
# ============================================================

spearman_figures = {}


for variable in visitor_vars:

    spearman_figures[
        variable
    ] = make_spearman_graph(
        variable
    )


# ============================================================
# 5. 그래프 출력
# ============================================================

for variable in visitor_vars:

    print()
    print("=" * 60)
    print(
        "Spearman:",
        variable
    )
    print("=" * 60)

    spearman_figures[
        variable
    ].show()


Spearman: 현지인 방문자 수_백만



Spearman: 외지인 방문자 수_백만



Spearman: 외국인 방문자 수_백만


In [90]:
# ============================================================
# 4단계. 연도 + 행정구 필터 테스트
# ============================================================

import plotly.graph_objects as go


# ============================================================
# 1. 테스트할 변수
# ============================================================

test_variable = "외국인 방문자 수_백만"


# ============================================================
# 2. 그래프 생성
# ============================================================

filter_fig = go.Figure()


# ============================================================
# 3. 연도 × 행정구별 trace 생성
# ============================================================

trace_info = []


for year in years:

    for district in districts:

        temp = analysis_df[
            (
                analysis_df["연도"]
                == year
            )
            &
            (
                analysis_df["행정구"]
                == district
            )
        ]


        if len(temp) == 0:

            continue


        # ----------------------------------------------------
        # hover 정보
        # ----------------------------------------------------

        hover_text = []


        for _, row in temp.iterrows():

            text = (

                "연도: "
                + str(row["연도"])

                + "<br>"

                + "행정구: "
                + str(row["행정구"])

                + "<br>"

                + "외국인 방문자 수: "
                + format(
                    row[test_variable],
                    ".3f"
                )
                + " 백만"

                + "<br>"

                + "폐업률: "
                + format(
                    row["폐업률"],
                    ".2f"
                )
                + "%"

            )

            hover_text.append(text)


        # ----------------------------------------------------
        # trace 추가
        # ----------------------------------------------------

        filter_fig.add_trace(

            go.Scatter(

                x=temp[
                    test_variable
                ],

                y=temp[
                    "폐업률"
                ],

                mode="markers",

                name=district,

                legendgroup=district,

                showlegend=(
                    year
                    ==
                    years[0]
                ),

                marker=dict(

                    size=20,

                    color=district_colors[
                        district
                    ],

                    symbol=year_symbols[
                        year
                    ],

                    line=dict(

                        color="black",

                        width=1

                    )

                ),

                text=hover_text,

                hovertemplate=
                    "%{text}"
                    +
                    "<extra></extra>"

            )

        )


        # ----------------------------------------------------
        # 각 trace의 연도/행정구 정보 저장
        # ----------------------------------------------------

        trace_info.append(

            {
                "year": str(year),
                "district": district
            }

        )


# ============================================================
# 4. 축 범위 고정
# ============================================================

xmin = analysis_df[
    test_variable
].min()

xmax = analysis_df[
    test_variable
].max()

x_margin = (
    xmax - xmin
) * 0.08


if x_margin == 0:

    x_margin = 1


ymin = analysis_df[
    "폐업률"
].min()

ymax = analysis_df[
    "폐업률"
].max()

y_margin = (
    ymax - ymin
) * 0.08


if y_margin == 0:

    y_margin = 1


# ============================================================
# 5. 그래프 Layout
# ============================================================

filter_fig.update_layout(

    title=(
        "필터 테스트 - 외국인 방문자 수 ↔ 폐업률"
    ),

    xaxis=dict(

        title="외국인 방문자 수_백만",

        range=[
            xmin - x_margin,
            xmax + x_margin
        ],

        autorange=False

    ),

    yaxis=dict(

        title="폐업률 (%)",

        range=[
            ymin - y_margin,
            ymax + y_margin
        ],

        autorange=False

    ),

    height=650,

    template="plotly_white",

    font=dict(

        family=(
            "Malgun Gothic, "
            "Noto Sans KR, "
            "Arial"
        )

    )

)


# ============================================================
# 6. 그래프 출력
# ============================================================

filter_fig.show()


# ============================================================
# 7. trace 정보 확인
# ============================================================

print()
print("=" * 60)
print("필터 테스트용 trace 확인")
print("=" * 60)

print(
    "전체 trace 수 :",
    len(trace_info)
)

print(
    "예상 trace 수 :",
    len(years) * len(districts)
)

print()

print(
    "첫 번째 trace :",
    trace_info[0]
)

print(
    "마지막 trace :",
    trace_info[-1]
)


필터 테스트용 trace 확인
전체 trace 수 : 48
예상 trace 수 : 48

첫 번째 trace : {'year': '2023', 'district': '강서구'}
마지막 trace : {'year': '2025', 'district': '해운대구'}


In [92]:
# ============================================================
# 4-1 수정
# Plotly 자체 Dropdown을 이용한 연도 필터
# ============================================================

import plotly.graph_objects as go


# ============================================================
# 1. 사용할 변수
# ============================================================

filter_variable = "외국인 방문자 수_백만"


# ============================================================
# 2. 그래프 생성
# ============================================================

year_filter_fig = go.Figure()


# ============================================================
# 3. 전체 데이터에서 행정구별 trace 생성
# ============================================================

for district in districts:

    temp = analysis_df[
        analysis_df["행정구"] == district
    ]


    # --------------------------------------------------------
    # 연도별로 하나의 trace를 만들기 때문에
    # 같은 행정구 안에서도 연도별 마커 모양을 다르게 설정
    # --------------------------------------------------------

    for year in years:

        year_temp = temp[
            temp["연도"] == year
        ]


        if len(year_temp) == 0:
            continue


        # ----------------------------------------------------
        # hover 정보
        # ----------------------------------------------------

        hover_text = []

        for _, row in year_temp.iterrows():

            text = (
                "연도: "
                + str(row["연도"])
                + "<br>"
                + "행정구: "
                + str(row["행정구"])
                + "<br>"
                + "외국인 방문자 수: "
                + format(
                    row[filter_variable],
                    ".3f"
                )
                + " 백만"
                + "<br>"
                + "폐업률: "
                + format(
                    row["폐업률"],
                    ".2f"
                )
                + "%"
            )

            hover_text.append(text)


        # ----------------------------------------------------
        # trace 추가
        # ----------------------------------------------------

        year_filter_fig.add_trace(

            go.Scatter(

                x=year_temp[
                    filter_variable
                ],

                y=year_temp[
                    "폐업률"
                ],

                mode="markers",

                name=district,

                legendgroup=district,

                showlegend=(
                    year == years[0]
                ),

                marker=dict(

                    size=20,

                    color=district_colors[
                        district
                    ],

                    symbol=year_symbols[
                        year
                    ],

                    line=dict(
                        color="black",
                        width=1
                    )

                ),

                text=hover_text,

                hovertemplate=(
                    "%{text}"
                    "<extra></extra>"
                )

            )

        )


# ============================================================
# 4. 전체 데이터 기준 축 범위
# ============================================================

xmin = analysis_df[
    filter_variable
].min()

xmax = analysis_df[
    filter_variable
].max()

x_margin = (
    xmax - xmin
) * 0.08

if x_margin == 0:
    x_margin = 1


ymin = analysis_df[
    "폐업률"
].min()

ymax = analysis_df[
    "폐업률"
].max()

y_margin = (
    ymax - ymin
) * 0.08

if y_margin == 0:
    y_margin = 1


# ============================================================
# 5. 연도별 표시 여부 배열 생성
# ============================================================

number_of_traces = len(
    year_filter_fig.data
)


# ------------------------------------------------------------
# 전체 보기
# ------------------------------------------------------------

all_visible = [
    True
    for _ in range(
        number_of_traces
    )
]


# ------------------------------------------------------------
# 연도별 표시 여부
# ------------------------------------------------------------

year_visibility = {}


for selected_year in years:

    visibility = []


    trace_index = 0


    for district in districts:

        for year in years:

            if trace_index >= number_of_traces:

                continue


            visibility.append(
                str(year)
                ==
                str(selected_year)
            )


            trace_index += 1


    year_visibility[
        str(selected_year)
    ] = visibility


# ============================================================
# 6. Dropdown 버튼 생성
# ============================================================

year_buttons = []


# ------------------------------------------------------------
# 전체
# ------------------------------------------------------------

year_buttons.append(

    dict(

        label="전체",

        method="update",

        args=[
            {
                "visible": all_visible
            },
            {
                "title": (
                    "외국인 방문자 수 ↔ 폐업률"
                    "<br>"
                    "<sup>전체 연도</sup>"
                )
            }
        ]

    )

)


# ------------------------------------------------------------
# 각 연도
# ------------------------------------------------------------

for selected_year in years:

    year_buttons.append(

        dict(

            label=str(
                selected_year
            ),

            method="update",

            args=[
                {
                    "visible":
                    year_visibility[
                        str(selected_year)
                    ]
                },
                {
                    "title": (
                        "외국인 방문자 수 ↔ 폐업률"
                        "<br>"
                        "<sup>"
                        + str(selected_year)
                        + "년"
                        + "</sup>"
                    )
                }
            ]

        )

    )


# ============================================================
# 7. 그래프 Layout
# ============================================================

year_filter_fig.update_layout(

    title=(
        "외국인 방문자 수 ↔ 폐업률"
        "<br>"
        "<sup>전체 연도</sup>"
    ),

    xaxis=dict(

        title="외국인 방문자 수_백만",

        range=[
            xmin - x_margin,
            xmax + x_margin
        ],

        autorange=False

    ),

    yaxis=dict(

        title="폐업률 (%)",

        range=[
            ymin - y_margin,
            ymax + y_margin
        ],

        autorange=False

    ),

    height=650,

    template="plotly_white",

    font=dict(

        family=(
            "Malgun Gothic, "
            "Noto Sans KR, "
            "Arial"
        )

    ),

    legend=dict(

        title="행정구"

    ),

    updatemenus=[

        dict(

            type="dropdown",

            direction="down",

            x=0.0,

            y=1.15,

            xanchor="left",

            yanchor="top",

            buttons=year_buttons,

            showactive=True

        )

    ]

)


# ============================================================
# 8. 그래프 출력
# ============================================================

year_filter_fig.show()

In [93]:
# ============================================================
# 4-2단계
# 연도 + 행정구 통합 필터
# ============================================================

import plotly.graph_objects as go


# ============================================================
# 1. 분석 변수
# ============================================================

filter_variable = "외국인 방문자 수_백만"


# ============================================================
# 2. 그래프 생성
# ============================================================

combined_fig = go.Figure()


# ============================================================
# 3. 데이터별 trace 생성
# ============================================================

trace_info_combined = []


for year in years:

    for district in districts:

        temp = analysis_df[
            (analysis_df["연도"] == year)
            &
            (analysis_df["행정구"] == district)
        ]


        if len(temp) == 0:
            continue


        # ----------------------------------------------------
        # Hover 정보
        # ----------------------------------------------------

        hover_text = []

        for _, row in temp.iterrows():

            text = (
                "연도: "
                + str(row["연도"])
                + "<br>"
                + "행정구: "
                + str(row["행정구"])
                + "<br>"
                + "외국인 방문자 수: "
                + format(
                    row[filter_variable],
                    ".3f"
                )
                + " 백만"
                + "<br>"
                + "폐업률: "
                + format(
                    row["폐업률"],
                    ".2f"
                )
                + "%"
            )

            hover_text.append(text)


        # ----------------------------------------------------
        # Trace 생성
        # ----------------------------------------------------

        combined_fig.add_trace(

            go.Scatter(

                x=temp[
                    filter_variable
                ].tolist(),

                y=temp[
                    "폐업률"
                ].tolist(),

                mode="markers",

                name=district,

                legendgroup=district,

                showlegend=(
                    year == years[0]
                ),

                marker=dict(

                    size=20,

                    color=district_colors[
                        district
                    ],

                    symbol=year_symbols[
                        year
                    ],

                    line=dict(
                        color="black",
                        width=1
                    )

                ),

                text=hover_text,

                hovertemplate=(
                    "%{text}"
                    "<extra></extra>"
                )

            )

        )


        trace_info_combined.append(
            {
                "year": str(year),
                "district": district
            }
        )


# ============================================================
# 4. 전체 데이터 기준 축 범위
# ============================================================

xmin = analysis_df[
    filter_variable
].min()

xmax = analysis_df[
    filter_variable
].max()

x_margin = (
    xmax - xmin
) * 0.08

if x_margin == 0:
    x_margin = 1


ymin = analysis_df[
    "폐업률"
].min()

ymax = analysis_df[
    "폐업률"
].max()

y_margin = (
    ymax - ymin
) * 0.08

if y_margin == 0:
    y_margin = 1


# ============================================================
# 5. 전체 표시
# ============================================================

number_of_traces = len(
    trace_info_combined
)


all_visible = [
    True
    for _ in range(
        number_of_traces
    )
]


# ============================================================
# 6. 필터 버튼 생성
# ============================================================

filter_buttons = []


# ============================================================
# 6-1. 전체
# ============================================================

filter_buttons.append(

    dict(

        label="전체",

        method="update",

        args=[

            {
                "visible": all_visible
            },

            {
                "title": (
                    "외국인 방문자 수 ↔ 폐업률"
                    "<br>"
                    "<sup>전체 데이터</sup>"
                )
            }

        ]

    )

)


# ============================================================
# 6-2. 연도별 전체
# ============================================================

for selected_year in years:

    visibility = []


    for info in trace_info_combined:

        visibility.append(

            info["year"]
            ==
            str(selected_year)

        )


    filter_buttons.append(

        dict(

            label=(
                str(selected_year)
                + "년 전체"
            ),

            method="update",

            args=[

                {
                    "visible": visibility
                },

                {
                    "title": (
                        "외국인 방문자 수 ↔ 폐업률"
                        "<br>"
                        "<sup>"
                        + str(selected_year)
                        + "년 전체 행정구"
                        + "</sup>"
                    )
                }

            ]

        )

    )


# ============================================================
# 6-3. 연도 + 행정구
# ============================================================

for selected_year in years:

    for selected_district in districts:

        visibility = []


        for info in trace_info_combined:

            visibility.append(

                (
                    info["year"]
                    ==
                    str(selected_year)
                )

                and

                (
                    info["district"]
                    ==
                    selected_district
                )

            )


        filter_buttons.append(

            dict(

                label=(
                    str(selected_year)
                    + "년 "
                    + selected_district
                ),

                method="update",

                args=[

                    {
                        "visible": visibility
                    },

                    {
                        "title": (
                            "외국인 방문자 수 ↔ 폐업률"
                            "<br>"
                            "<sup>"
                            + str(selected_year)
                            + "년 "
                            + selected_district
                            + "</sup>"
                        )
                    }

                ]

            )

        )


# ============================================================
# 7. 그래프 Layout
# ============================================================

combined_fig.update_layout(

    title=(
        "외국인 방문자 수 ↔ 폐업률"
        "<br>"
        "<sup>전체 데이터</sup>"
    ),

    xaxis=dict(

        title="외국인 방문자 수_백만",

        range=[
            xmin - x_margin,
            xmax + x_margin
        ],

        autorange=False

    ),

    yaxis=dict(

        title="폐업률 (%)",

        range=[
            ymin - y_margin,
            ymax + y_margin
        ],

        autorange=False

    ),

    height=700,

    template="plotly_white",

    font=dict(

        family=(
            "Malgun Gothic, "
            "Noto Sans KR, "
            "Arial"
        )

    ),

    legend=dict(

        title="행정구"

    ),

    updatemenus=[

        dict(

            type="dropdown",

            direction="down",

            x=0.0,

            y=1.15,

            xanchor="left",

            yanchor="top",

            buttons=filter_buttons,

            showactive=True,

            bgcolor="white",

            bordercolor="gray"

        )

    ]

)


# ============================================================
# 8. 그래프 출력
# ============================================================

combined_fig.show()